<a href="https://colab.research.google.com/github/Shivabairy005/Deep-Learning-Lab/blob/main/DL_lab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np

train_gen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(rescale=1.0 / 255)

train_data = train_gen.flow_from_directory(
    "chest_xray/train",
    target_size=(150, 150),
    batch_size=32,
    class_mode="binary"
)

val_data = val_gen.flow_from_directory(
    "chest_xray/val",
    target_size=(150, 150),
    batch_size=32,
    class_mode="binary"
)

def build_model(dropout_rate=0.0, l2_value=0.0, add_noise=False):
    model = tf.keras.Sequential()

    if add_noise:
        model.add(layers.GaussianNoise(0.1, input_shape=(150, 150, 3)))
    else:
        model.add(layers.Input(shape=(150, 150, 3)))

    model.add(
        layers.Conv2D(
            32,
            (3, 3),
            activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(l2_value)
        )
    )
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Flatten())

    if dropout_rate > 0:
        model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

baseline = build_model()
baseline.fit(train_data, epochs=10, validation_data=val_data)

l2_model = build_model(l2_value=0.001)
l2_model.fit(train_data, epochs=10, validation_data=val_data)

drop_model = build_model(dropout_rate=0.5)
drop_model.fit(train_data, epochs=10, validation_data=val_data)

noise_model = build_model(add_noise=True)
noise_model.fit(train_data, epochs=10, validation_data=val_data)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

early_model = build_model(dropout_rate=0.3)
early_model.fit(
    train_data,
    epochs=20,
    validation_data=val_data,
    callbacks=[early_stop]
)

models = []
for i in range(3):
    m = build_model(dropout_rate=0.3)
    m.fit(train_data, epochs=5, validation_data=val_data)
    models.append(m)

val_steps = len(val_data)
val_data.reset()
preds = np.mean([m.predict(val_data, steps=val_steps) for m in models], axis=0)
pred_labels = (preds > 0.5).astype(int).flatten()

val_data.reset()
true_labels = val_data.classes

print("Ensemble Accuracy:", np.mean(pred_labels == true_labels))

FileNotFoundError: [Errno 2] No such file or directory: 'chest_xray/train'